> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 高级上下文工程：智能体的大脑

欢迎来到 Semantica 上下文工程的**大师课**。本 notebook 演示如何为 AI 智能体构建生产级记忆系统。

会一结束就忘光的简单聊天机器人不算数，**有上下文感知的智能体**需要：
*   **长期记忆**：回忆几周前的事实
*   **结构化知识**：理解实体（人物、项目、主题）如何关联
*   **混合检索**：模糊的文本搜索 + 精确的图遍历

### 先看主线

19 篇给智能体装了"海马体"（入门、内存跑），这一篇升到**生产级大脑**：真后端（FAISS/Neo4j）、GraphRAG 调参、手动教关系、连点成线的检索魔术。七节四段：

**装好后端（第 1–2 节）—— 图书馆与地图**

向量库存*内容*（图书馆）、图库存*连接*（地图）——统一接口，后端随换。

**大脑上线（第 3–4 节）—— 编排与喂知识**

`AgentContext` 是中枢：图扩展开关、跳数、混合权重（`hybrid_alpha`：0 纯向量、1 纯图）。然后喂两类知识——情节记忆（对话）与语义知识（文档，自动抽实体）。

**教关系（第 5 节）—— 手动注入**

自动抽取不够表达业务逻辑时，`build_graph` 手动教："Alice MANAGES Project Apollo"。

**连点成线（第 6 节）与保养（第 7 节）**

第 6 节是题眼：问"谁负责 Python Web 框架项目"——项目描述里**没有 Alice 的名字**，但向量找到文档、图找到人。第 7 节收尾：历史、体检、修剪旧记忆。

## 学习目标

本走读中，我们将：
1.  **初始化生产级存储**：用真正的**向量库**（FAISS）和**图库**（Neo4j）替换玩具示例
2.  **构建智能体上下文**：配置编排记忆的中枢大脑
3.  **摄取知识**：存复杂文档、自动抽实体
4.  **注入关系**：手动教智能体世界中的连接
5.  **执行 GraphRAG**：跑"跳"过知识图谱的高级查询，找到标准 RAG 漏掉的答案
6.  **管理生命周期**：学修剪旧记忆、保持系统健康

---

## 1. 安装

开始只需安装包：

```bash
pip install semantica
```

In [ ]:
!pip install -qU semantica 

In [ ]:
import sys
import os
import time
from typing import Any, List, Dict, Optional

# Add project root to path to import semantica
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))

# Core Imports
from semantica.context import AgentContext, ContextGraph, AgentMemory
from semantica.vector_store import VectorStore
from semantica.graph_store import GraphStore

print("Libraries imported successfully.")

---

## 2. 初始化存储后端

现在连接持久化存储层。Semantica 用统一接口把它们抽象起来——换后端（比如 FAISS 换 Weaviate）不用改应用逻辑。

### 向量库（图书馆）
保存记忆和文档的*内容*，按语义索引。

In [ ]:
try:
    # Initialize FAISS Vector Store
    # You can also use: backend="weaviate", backend="qdrant", etc.
    vs = VectorStore(backend="faiss", dimension=768)
    print("VectorStore initialized (Backend: FAISS)")
except ImportError:
    print("FAISS not installed. Using in-memory fallback (not persistent).")
    vs = VectorStore(backend="inmemory", dimension=768)
except Exception as e:
    print(f"VectorStore Error: {e}")
    vs = None

### 图库（地图）
保存实体之间的*连接*——推理的关键。需要运行中的 Neo4j；连不上会自动降级到内存版 ContextGraph。

In [ ]:
try:
    # Initialize Neo4j Graph Store
    # Ensure your Docker container is running!
    gs = GraphStore(
        backend="neo4j",
        uri="bolt://localhost:7687",
        user="neo4j",
        password="password"
    )
    
    # Test connection
    if gs.connect():
        print("GraphStore connected (Backend: Neo4j)")
    else:
        raise ConnectionError("Could not connect to Neo4j")

except Exception as e:
    print(f"GraphStore Connection Failed: {e}")
    print("   Switching to in-memory ContextGraph (Non-persistent fallback)")
    gs = ContextGraph() # Fallback implementation

---

## 3. 智能体上下文

`AgentContext` 是高层编排器，坐在向量库和图库之上，管理信息流。

**GraphRAG 配置：**
*   `use_graph_expansion=True`：检索时不只看文档，还看它的邻居
*   `max_expansion_hops=2`：遍历多远？（如 A → B → C）
*   `hybrid_alpha=0.6`：权重。0.0 纯向量，1.0 纯图，0.6 略偏图。

In [ ]:
if vs:
    context = AgentContext(
        vector_store=vs,
        knowledge_graph=gs,
        retention_days=90,          # Remember things for 3 months
        use_graph_expansion=True,   # Enable GraphRAG
        max_expansion_hops=2,       # 2-Hop reasoning
        hybrid_alpha=0.6            # Balanced retrieval
    )
    print("Agent Context is online and ready.")
else:
    print("Cannot proceed without VectorStore.")

---

## 4. 摄取：教智能体

可以存不同类型的信息。系统够聪明，能区分对话记忆和事实文档。

### A. 情节记忆（对话）
交互的原始日志，提供"个人"历史。

In [ ]:
user_id = "user_123"
session_id = "session_alpha"

# Store a user preference
mem_id = context.store(
    content="I am working on a new project called 'Project Apollo' which uses Python and React.",
    conversation_id=session_id,
    user_id=user_id,
    metadata={"type": "user_preference"}
)
print(f"Memory Stored: {mem_id}")

### B. 语义知识（文档）
喂文档时，我们要**抽实体**并**建立链接**。

*（注：真实部署用 LLM 解析实体；这里用 context 模块的原生抽取能力。）*

In [ ]:
documents = [
    {
        "content": "Project Apollo is a next-gen web framework designed for high scalability.",
        "metadata": {"source": "internal_wiki", "category": "projects"}
    },
    {
        "content": "Python 3.12 introduces significant performance improvements for async workloads.",
        "metadata": {"source": "tech_news", "category": "languages"}
    }
]

# Store documents and trigger graph build
stats = context.store(
    documents,
    extract_entities=True,      # Extract entities from text
    extract_relationships=True, # Infer relationships
    link_entities=True          # Connect to existing graph nodes
)

print("Knowledge Ingestion Stats:", stats)

---

## 5. 图工程：手动注入

自动抽取有时不够——你想强制表达特定的业务逻辑或关系。用 `build_graph` 手动注入节点和边。

**我们要定义：**
*   **用户**（Alice）
*   **角色**（Admin）
*   **项目**（Apollo）
*   **关系**：Alice *MANAGES* Project Apollo。

In [ ]:
# 1. Define Nodes
entities = [
    {"id": "alice", "type": "PERSON", "text": "Alice", "properties": {"role": "Admin"}},
    {"id": "project_apollo", "type": "PROJECT", "text": "Project Apollo"},
    {"id": "python", "type": "TECH", "text": "Python"},
    {"id": "react", "type": "TECH", "text": "React"}
]

# 2. Define Edges (The Knowledge)
relationships = [
    {"source": "alice", "target": "project_apollo", "type": "MANAGES", "weight": 1.0},
    {"source": "project_apollo", "target": "python", "type": "USES_TECH", "weight": 1.0},
    {"source": "project_apollo", "target": "react", "type": "USES_TECH", "weight": 1.0}
]

# 3. Inject into Graph
graph_stats = context.build_graph(
    entities=entities,
    relationships=relationships
)

print("Manual Graph Build Complete:", graph_stats)

### 可视化图的逻辑
直接查图，看看"Project Apollo"长什么样。

In [ ]:
# Helper to print graph neighbors
def inspect_node(node_id):
    if hasattr(gs, "get_neighbors"):
        neighbors = gs.get_neighbors(node_id)
        print(f"\nNeighbors of '{node_id}':")
        for n in neighbors:
            # Handle different return formats between stores
            rel_type = n.get('relationship') or n.get('type') or 'linked'
            target = n.get('id') or n.get('node_id')
            print(f"   └── [{rel_type}] ──> {target}")
    else:
        print("Graph store does not support neighbor inspection.")

inspect_node("project_apollo")

---

## 6. 混合检索（GraphRAG）

见证魔术：问一个需要连点成线的问题。

**查询**：*"谁负责 Python Web 框架项目？"*

**逻辑流：**
1.  **向量搜索**：找到 "Project Apollo"（描述里说是 Web 框架）
2.  **图扩展**：在图里看 "Project Apollo" 的邻居
3.  **发现**：看到 `(Alice)-[MANAGES]->(Project Apollo)`
4.  **结果**：返回 Alice——尽管项目描述文本里根本没提她的名字！

In [ ]:
query = "Who is responsible for the Python web framework project?"
print(f"Asking: '{query}'...\n")

results = context.retrieve(
    query,
    max_results=3,
    use_graph=True,         # Vital for finding Alice
    expand_graph=True,      # Hop to neighbors
    include_entities=True   # Return structured entity data
)

print(f"Retrieved {len(results)} context items:\n")

for i, res in enumerate(results, 1):
    print(f"{i}. [Score: {res['score']:.2f}] {res['content'][:120]}...")
    
    # Did we find graph connections?
    if 'related_entities' in res and res['related_entities']:
        print("   Graph Insights:")
        for ent in res['related_entities'][:3]:
            print(f"      - {ent.get('text', 'Entity')} ({ent.get('type', 'Unknown')})")
    print("")

---

## 7. 生命周期管理

生产系统需要保养：查历史、看健康度、修剪旧数据。

### 对话历史

In [ ]:
# Get recent chat history for context window
history = context.conversation(
    conversation_id=session_id,
    limit=5
)

print(f"Chat History for {session_id}:")
for msg in history:
    print(f" - {msg['content']}")

### 系统健康与统计

In [ ]:
stats = context.stats()
print("System Vital Signs:")
print(f"   - Total Memories: {stats.get('total_items', 0)}")
print(f"   - Graph Nodes:    {stats.get('graph_stats', {}).get('node_count', 'N/A')}")
print(f"   - Graph Edges:    {stats.get('graph_stats', {}).get('edge_count', 'N/A')}")

## 小结

### 回顾主线

大脑装完：**后端**（图书馆与地图——内容与连接各就各位）、**中枢**（AgentContext 调 GraphRAG 的三个旋钮）、**喂知识**（对话与文档两类）、**教关系**（手动注入业务逻辑）、**连点成线**（向量找文档、图找人——名字不在文本里也能找到）、**保养**（历史、体检、修剪）。19 篇的海马体，升级成了能上生产的大脑。

你已用 Semantica 的生产模块成功构建了一个**有上下文感知的智能体**。

**关键成果：**
1.  **持久化**：换上 FAISS 和 Neo4j 做真实存储
2.  **GraphRAG**：演示了图关系如何提升检索精度
3.  **实体注入**：手动教智能体业务关系

这套架构可以扩展到百万级向量与图节点。